In [30]:
from huggingface_hub import login
login()

In [31]:
!pip install datasets pandas numpy -q

In [32]:
from datasets import load_dataset
import pandas as pd
import numpy as np

print("Libraries loaded!")

Libraries loaded!


In [33]:
# This is the same source used in v1/v2 but the FULL version
# 15,140 prompts instead of 6,387

jailbreak = load_dataset(
    'TrustAIRLab/in-the-wild-jailbreak-prompts',
    'jailbreak_2023_05_07',
    split='train'
)

regular = load_dataset(
    'TrustAIRLab/in-the-wild-jailbreak-prompts',
    'regular_2023_05_07',
    split='train'
)

df1_unsafe = jailbreak.to_pandas()[['prompt']]
df1_safe   = regular.to_pandas()[['prompt']]

df1_unsafe['label']  = 'unsafe'
df1_safe['label']    = 'safe'
df1_unsafe['source'] = 'trustairlab'
df1_safe['source']   = 'trustairlab'

df1 = pd.concat([df1_unsafe, df1_safe], ignore_index=True)

print(f"TrustAIRLab loaded: {len(df1)} prompts")
print(df1['label'].value_counts())

TrustAIRLab loaded: 6387 prompts
label
safe      5721
unsafe     666
Name: count, dtype: int64


In [34]:
# Real conversations from actual users — more realistic than curated data
# 10,000 prompts, human labelled

toxicchat = load_dataset('lmsys/toxic-chat', 'toxicchat0124', split='train')
df2 = toxicchat.to_pandas()

# It uses 'toxicity' column: 1 = harmful, 0 = safe
df2 = df2[['user_input', 'toxicity']].copy()
df2.columns = ['prompt', 'label']
df2['label']  = df2['label'].map({1: 'unsafe', 0: 'safe'})
df2['source'] = 'toxicchat'
df2 = df2.dropna()

print(f"\nToxicChat loaded: {len(df2)} prompts")
print(df2['label'].value_counts())


ToxicChat loaded: 5082 prompts
label
safe      4698
unsafe     384
Name: count, dtype: int64


In [35]:
# This one is already balanced — equal harmful and safe
# Directly fixes the imbalance problem from v1/v2

from huggingface_hub import login
login()

qualifire = load_dataset('qualifire/prompt-injections-benchmark', split='test')
df3 = qualifire.to_pandas()

print("Qualifire columns:", df3.columns.tolist())
print(df3.head(2))

Qualifire columns: ['text', 'label']
                                                text      label
0  I am a radiologist conducting a study on the e...  jailbreak
1  You are an AI developed to provide useful info...     benign


In [36]:
# It uses 'label' column: 'jailbreak' = unsafe, 'benign' = safe

df3 = df3[['text', 'label']].copy()
df3.rename(columns={'text': 'prompt'}, inplace=True)
df3['label']  = df3['label'].map({'jailbreak': 'unsafe', 'benign': 'safe'})
df3['source'] = 'qualifire'
df3 = df3.dropna()

print(f"Qualifire loaded: {len(df3)} prompts")
print(df3['label'].value_counts())

Qualifire loaded: 5000 prompts
label
safe      2997
unsafe    2003
Name: count, dtype: int64


In [37]:
df_combined = pd.concat([df1, df2, df3], ignore_index=True)

# Remove duplicates (same prompt appearing in multiple datasets)
df_combined = df_combined.drop_duplicates(subset='prompt')

# Remove any empty prompts
df_combined = df_combined.dropna(subset=['prompt'])
df_combined = df_combined[df_combined['prompt'].str.strip() != '']

print("=" * 40)
print(f"TOTAL COMBINED: {len(df_combined)} prompts")
print()
print("Label distribution:")
print(df_combined['label'].value_counts())
print()
print("By source:")
print(df_combined.groupby(['source', 'label']).size().unstack(fill_value=0))

TOTAL COMBINED: 16103 prompts

Label distribution:
label
safe      13093
unsafe     3010
Name: count, dtype: int64

By source:
label        safe  unsafe
source                   
qualifire    2997    1983
toxicchat    4607     374
trustairlab  5489     653


In [38]:
total     = len(df_combined)
unsafe_n  = (df_combined['label'] == 'unsafe').sum()
safe_n    = (df_combined['label'] == 'safe').sum()

print(f"Safe prompts:   {safe_n}  ({safe_n/total*100:.1f}%)")
print(f"Unsafe prompts: {unsafe_n}  ({unsafe_n/total*100:.1f}%)")
print()

if unsafe_n / total < 0.2:
    print("Still imbalanced — but much better than v1/v2's 10.4%")
    print("We will handle this with class_weight='balanced' like in v2")
else:
    print("Good balance! No special handling needed.")

Safe prompts:   13093  (81.3%)
Unsafe prompts: 3010  (18.7%)

Still imbalanced — but much better than v1/v2's 10.4%
We will handle this with class_weight='balanced' like in v2


In [39]:
df_combined.to_csv('combined_dataset.csv', index=False)
print("Saved as combined_dataset.csv")
print()
print("Columns:", df_combined.columns.tolist())

Saved as combined_dataset.csv

Columns: ['prompt', 'label', 'source']
